# Gerador de Dados Customizado

Use este notebook para gerar e inspecionar datasets de forma interativa.
Os arquivos `.npz` gerados aqui podem ser usados diretamente no treinamento:

```bash
python pinn_main.py --dataset data_synthetic/meu_experimento_20260218.npz
```

---
**Experimentos disponíveis:**

| case | f(x,t) | Descricao |
|---|---|---|
| `example_1` | `sin(pi*x) * exp(-t)` | Baseline suave |
| `example_2` | `(1 + 0.5*sin(2*pi*x)) * exp(-2t)` | Campo variavel |
| `example_3` | `x*(1-x) * sin(2*pi*t)` | Variacao temporal |
| `example_4` | descontinuo em x=0.5 | Descontinuidade |
| `example_5` | igual ao exp_1 + ruido | Robustez a ruido |

In [ ]:
import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['font.family'] = 'serif'
matplotlib.rcParams['mathtext.fontset'] = 'cm'
matplotlib.rcParams['font.size'] = 11

# Garantir que os modulos do projeto estao no path
project_root = os.path.abspath('')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from synthetic_data_generator import SyntheticDataGenerator, load_dataset
from analytical_solutions import u_real, f_real, EXPERIMENT_DESCRIPTIONS

print('Modulos carregados com sucesso.')

---
## 1. Configuracao do Experimento

Edite os parametros abaixo e execute a celula.

In [ ]:
# ================================================================
#  PARAMETROS -- edite aqui
# ================================================================

CASE         = 'example_1'   # 'example_1' ... 'example_5'
DOMAIN       = [0, 1, 0, 1]  # [x_min, x_max, t_min, t_max]
ALPHA        = 1.0            # coeficiente de difusao (conhecido)
NOISE_LEVEL  = 0.0            # 0.0 = sem ruido  |  0.05 = 5%

# Resolucao dos conjuntos de dados
N_TEST_X   = 200   # pontos de teste em x
N_TEST_T   = 200   # pontos de teste em t
N_TRAIN_X  = 150   # pontos PDE em x
N_TRAIN_T  = 150   # pontos PDE em t
N_BC       = 1000  # pontos de contorno/CI
N_F_OBS    = 200   # observacoes esparsas de f(x,t)

OUTPUT_DIR  = 'data_synthetic'   # pasta de saida
FILE_PREFIX = f'custom_{CASE}'   # prefixo do arquivo .npz

# ================================================================

config = {
    'experiment': {
        'case':        CASE,
        'domain':      DOMAIN,
        'alpha':       ALPHA,
        'noise_level': NOISE_LEVEL,
    }
}

desc = EXPERIMENT_DESCRIPTIONS.get(CASE, {})
print(f"Experimento : {desc.get('name', CASE)}")
print(f"f(x,t)      : {desc.get('f_formula', 'N/A')}")
print(f"u(x,t)      : {desc.get('u_formula', 'N/A')}")
print(f"Alpha       : {ALPHA}")
print(f"Ruido       : {NOISE_LEVEL*100:.1f}%")
print(f"Notas       : {desc.get('notes', '')}")

---
## 2. Inspecionar as Solucoes Verdadeiras

Antes de salvar, visualize `u(x,t)` e `f(x,t)` para confirmar que o experimento esta correto.

In [ ]:
gen = SyntheticDataGenerator(config)

# Malha de preview (resolucao menor para rapidez)
N_PREVIEW = 100
u_mesh, f_mesh, _, _, _, X, T = gen.get_test_dataset(N_PREVIEW, N_PREVIEW)

X_np = X.numpy(); T_np = T.numpy()
u_np = u_mesh.numpy(); f_np = f_mesh.numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# u(x,t)
c1 = axes[0].contourf(X_np, T_np, u_np, levels=20, cmap='viridis')
axes[0].set_title(r'Solucao verdadeira $u(x,t)$', fontsize=13)
axes[0].set_xlabel(r'$x$'); axes[0].set_ylabel(r'$t$')
plt.colorbar(c1, ax=axes[0], format='%.3f')

# f(x,t)
vmax = max(abs(f_np.min()), abs(f_np.max())) + 1e-12
lvls = np.linspace(-vmax, vmax, 21)
c2 = axes[1].contourf(X_np, T_np, f_np, levels=lvls, cmap='RdBu_r', extend='both')
axes[1].set_title(r'Termo fonte verdadeiro $f(x,t)$  [OBJETIVO]', fontsize=13)
axes[1].set_xlabel(r'$x$'); axes[1].set_ylabel(r'$t$')
plt.colorbar(c2, ax=axes[1], format='%.3f')

fig.suptitle(f"{desc.get('name', CASE)} | noise={NOISE_LEVEL*100:.0f}%", fontsize=12)
plt.tight_layout()
plt.show()

print(f"u in [{u_np.min():.4f}, {u_np.max():.4f}]")
print(f"f in [{f_np.min():.4f}, {f_np.max():.4f}]")

---
## 3. Inspecionar os Pontos de Treinamento

In [ ]:
x_pde          = gen.get_pde_dataset(N_TRAIN_X, N_TRAIN_T)
x_bc,   y_bc   = gen.get_bc_dataset(N_BC)
x_f_obs, f_obs = gen.get_source_observations(N_F_OBS)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].scatter(x_pde[:, 0], x_pde[:, 1], s=1, alpha=0.4, color='steelblue')
axes[0].set_title(f'Pontos PDE  (N={x_pde.shape[0]})', fontsize=12)
axes[0].set_xlabel(r'$x$'); axes[0].set_ylabel(r'$t$')

sc = axes[1].scatter(x_bc[:, 0], x_bc[:, 1], c=y_bc[:, 0],
                     s=5, alpha=0.7, cmap='viridis')
axes[1].set_title(f'Contorno/CI  (N={x_bc.shape[0]})', fontsize=12)
axes[1].set_xlabel(r'$x$'); axes[1].set_ylabel(r'$t$')
plt.colorbar(sc, ax=axes[1], label='u observado')

sf = axes[2].scatter(x_f_obs[:, 0], x_f_obs[:, 1], c=f_obs[:, 0],
                     s=10, alpha=0.8, cmap='RdBu_r')
axes[2].set_title(f'Obs. de f(x,t)  (N={x_f_obs.shape[0]})', fontsize=12)
axes[2].set_xlabel(r'$x$'); axes[2].set_ylabel(r'$t$')
plt.colorbar(sf, ax=axes[2], label='f observado')

plt.tight_layout()
plt.show()

---
## 4. Slices 1D para Inspecao Detalhada

In [ ]:
# Regenerar com resolucao completa para os slices
u_full, f_full, _, _, _, X_full, T_full = gen.get_test_dataset(N_TEST_X, N_TEST_T)
u_full = u_full.numpy(); f_full = f_full.numpy()
X_full = X_full.numpy(); T_full = T_full.numpy()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# f em t fixo
for frac, ax, label in [(0.25, axes[0,0], 't=0.25'), (0.75, axes[0,1], 't=0.75')]:
    ti = int(frac * (N_TEST_T - 1))
    ax.plot(X_full[:, 0], u_full[:, ti], 'b-',  lw=2, label=r'$u(x,t)$')
    ax.plot(X_full[:, 0], f_full[:, ti], 'r--', lw=2, label=r'$f(x,t)$')
    ax.set_title(f'Slice em {label}', fontsize=12)
    ax.set_xlabel(r'$x$'); ax.legend(); ax.grid(True, alpha=0.3)

# u e f em x fixo
for frac, ax, label in [(0.25, axes[1,0], 'x=0.25'), (0.5, axes[1,1], 'x=0.50')]:
    xi = int(frac * (N_TEST_X - 1))
    ax.plot(T_full[0, :], u_full[xi, :], 'b-',  lw=2, label=r'$u(x,t)$')
    ax.plot(T_full[0, :], f_full[xi, :], 'r--', lw=2, label=r'$f(x,t)$')
    ax.set_title(f'Slice em {label}', fontsize=12)
    ax.set_xlabel(r'$t$'); ax.legend(); ax.grid(True, alpha=0.3)

fig.suptitle('Slices 1D -- u(x,t) vs f(x,t)', fontsize=13)
plt.tight_layout()
plt.show()

---
## 5. Salvar o Dataset

Execute esta celula para salvar o `.npz`. O caminho sera exibido e pode ser passado diretamente para o treinamento.

In [ ]:
saved_path = gen.save_dataset(
    output_dir=OUTPUT_DIR,
    prefix=FILE_PREFIX,
    N_test_x=N_TEST_X,
    N_test_t=N_TEST_T,
    N_train_x=N_TRAIN_X,
    N_train_t=N_TRAIN_T,
    N_bc=N_BC,
    N_f_obs=N_F_OBS,
)

print()
print('=' * 60)
print('Dataset salvo!')
print(f'  Arquivo : {saved_path}')
print()
print('Para treinar com este dataset:')
print(f'  python pinn_main.py --dataset {saved_path}')
print('=' * 60)

---
## 6. (Opcional) Verificar o Dataset Salvo

Carrega o arquivo que acabou de ser salvo e confirma que os tensors estao corretos.

In [ ]:
ds = load_dataset(saved_path)

print('Conteudo do dataset:')
print(f"  case        : {ds['case']}")
print(f"  alpha       : {ds['alpha']}")
print(f"  noise_level : {ds['noise_level']}")
print()
for key in ['x_pde', 'x_bc', 'y_bc', 'x_test', 'u_test', 'f_test',
            'x_f_obs', 'f_obs', 'X', 'T', 'u_mesh', 'f_mesh']:
    t = ds[key]
    print(f"  {key:<12}: shape={tuple(t.shape)}  dtype={t.dtype}  "
          f"range=[{t.min():.3f}, {t.max():.3f}]")

---
## 7. (Opcional) Geracao em Lote de Multiplos Experimentos

Para gerar varios datasets de uma so vez sem sair do notebook.

In [ ]:
BATCH_EXPERIMENTS = [
    {'case': 'example_1', 'noise_level': 0.0,  'prefix': 'batch_exp1'},
    {'case': 'example_2', 'noise_level': 0.0,  'prefix': 'batch_exp2'},
    {'case': 'example_5', 'noise_level': 0.01, 'prefix': 'batch_exp5_noise01'},
    {'case': 'example_5', 'noise_level': 0.05, 'prefix': 'batch_exp5_noise05'},
    {'case': 'example_5', 'noise_level': 0.10, 'prefix': 'batch_exp5_noise10'},
]

BATCH_OUTPUT_DIR = 'data_synthetic'

# Parametros de resolucao compartilhados
BATCH_PARAMS = dict(
    N_test_x=200, N_test_t=200,
    N_train_x=150, N_train_t=150,
    N_bc=1000, N_f_obs=200,
)

batch_paths = []
for exp in BATCH_EXPERIMENTS:
    cfg = {
        'experiment': {
            'case':        exp['case'],
            'domain':      [0, 1, 0, 1],
            'alpha':       1.0,
            'noise_level': exp['noise_level'],
        }
    }
    g = SyntheticDataGenerator(cfg)
    p = g.save_dataset(output_dir=BATCH_OUTPUT_DIR,
                       prefix=exp['prefix'],
                       **BATCH_PARAMS)
    batch_paths.append(p)

print()
print('Datasets gerados:')
for p in batch_paths:
    print(f'  python pinn_main.py --dataset {p}')